# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [152]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [153]:
df = pd.read_csv("data/AviationData.csv", encoding="utf-8", encoding_errors="ignore")
df.sample(10)
df.columns
# df.isna().sum()
# df.shape
# df.info()
df.describe()

/tmp/ipykernel_400920/2052188250.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="utf-8", encoding_errors="ignore")


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


- Our dataset has 88,889 rows and 31 columns
- There are 5 numerical columns and 26 categorical columns
- There are missing values in the dataset 
- There's a jet with zero engines

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [154]:
# clean the column names
df.columns = df.columns.str.lower().str.replace(".", "_")

In [155]:
df.columns

Index(['event_id', 'investigation_type', 'accident_number', 'event_date',
       'location', 'country', 'latitude', 'longitude', 'airport_code',
       'airport_name', 'injury_severity', 'aircraft_damage',
       'aircraft_category', 'registration_number', 'make', 'model',
       'amateur_built', 'number_of_engines', 'engine_type', 'far_description',
       'schedule', 'purpose_of_flight', 'air_carrier', 'total_fatal_injuries',
       'total_serious_injuries', 'total_minor_injuries', 'total_uninjured',
       'weather_condition', 'broad_phase_of_flight', 'report_status',
       'publication_date'],
      dtype='str')

In [156]:
df.aircraft_category.unique()

<StringArray>
[                nan,          'Airplane',        'Helicopter',
            'Glider',           'Balloon',         'Gyrocraft',
        'Ultralight',           'Unknown',             'Blimp',
      'Powered-Lift',      'Weight-Shift', 'Powered Parachute',
            'Rocket',              'WSFT',               'UNK',
              'ULTR']
Length: 16, dtype: str

In [157]:
# client is only interested in category 'airplanes'
print(f"Before filtering airplanes: {len(df)}")
df = df[df.aircraft_category == "Airplane"]
print(f"After filtering airplanes: {len(df)}")

Before filtering airplanes: 88889
After filtering airplanes: 27617


In [158]:
# getting only aircraft with a lifetime not more than 40yrs
df.event_date = df.event_date.astype('datetime64[s]')
df = df[df.event_date.dt.year >= 1983]

In [159]:
# dropping amateur_builds since client is interested in professional Builds only
print(f"Before amateur filter: {len(df)}")
df = df[df.amateur_built == "No"]
print(f"After amateur filter: {len(df)}")

Before amateur filter: 24441
After amateur filter: 21447


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [160]:
# Create total passengers column using injuries columns
df["total_passengers"] = (
    df.total_fatal_injuries.fillna(0) +
    df.total_minor_injuries.fillna(0) +
    df.total_serious_injuries.fillna(0) +
    df.total_uninjured.fillna(0)
) 

# calculate total number of fatal and serious injuries
df['total_serious_fatal'] = (df.total_fatal_injuries + df.total_serious_injuries)
# metric for fatal/serious injury
df["injury_rate"] = df.total_serious_fatal / df.total_passengers

df.loc[df["total_passengers"] == 0, "injury_rate"] = None

- in injury_rate:
- 0 -> no serious/fatal injuries
- 1 -> everyone on board had serious/fatal injuries
- between 0 and 1 -> a proportion was affected

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [161]:
# check for unique values to understand levels of damage
df.aircraft_damage.unique()

# merge the nan with Unknown
df.aircraft_damage = df.aircraft_damage.fillna("Unknown")

In [162]:
# Create a column to track whether the aircraft was destroyed or not 
df["aircraft_destroyed"] = (df.aircraft_damage == "Destroyed").astype(int)

In [163]:
df.aircraft_destroyed.sample(5)

83341    0
75174    1
58464    0
35933    0
62850    0
Name: aircraft_destroyed, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

#### Make Column Cleaning Tasks

- Standardize text formatting (lowercase or title case)
- Remove leading and trailing whitespace
- Handle missing values

In [164]:
# converting to title case and removing white spaces
df.make = df.make.str.strip().str.title()

In [165]:
# filling null values
df.make = df.make.fillna("Unknown")

In [166]:
# Makes with reasonable number of atleast 50 record
# Count occurrence of each
make_count = df.make.value_counts()
# filter ones with more than 50 records
valid_makes = make_count[make_count >= 50].index

# subset df to remain with valid makes 
df = df[df.make.isin(valid_makes)]

In [167]:
# check results
df.make.value_counts()

make
Cessna                            7146
Piper                             3989
Beech                             1431
Boeing                            1264
Mooney                             363
Airbus                             243
Cirrus Design Corp                 220
Bellanca                           219
Air Tractor Inc                    219
Maule                              215
Air Tractor                        206
Aeronca                            200
Champion                           158
Embraer                            153
Grumman                            147
Luscombe                           141
Cirrus                             137
Stinson                            129
Mcdonnell Douglas                  108
North American                     106
Dehavilland                         95
Taylorcraft                         93
Aero Commander                      90
Aviat Aircraft Inc                  76
Socata                              75
Diamond Aircraft Ind

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [168]:
# get rid of NaNs
df = df.dropna(subset=["model"])

In [169]:
df.shape

(17879, 35)

In [170]:
# Inspect the column and counts for each model/make. Are model labels unique to each make? 
df.groupby("model")["make"].nunique().sort_values(ascending=False).sample(3)

model
BEE DEE M-4-210    1
150                1
6                  1
Name: make, dtype: int64

In [171]:
# create a derived column that is a unique identifier for a given plane type.
# clean both columns first
df.make = df.make.str.strip().str.title()
df.model = df.model.str.strip().str.title()

# Create new column
df["aircraft_type"] = df.make + " " + df.model

In [172]:
df[["make", "model", "aircraft_type"]].head()

,make,model,aircraft_type
4150,Boeing,747,Boeing 747
4171,Piper,Pa-28-140,Piper Pa-28-140
4285,De Havilland,Dhc-6,De Havilland Dhc-6
6760,Boeing,727-200,Boeing 727-200
6806,Beech,C35,Beech C35


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [173]:
# start by standadizing text 
df.engine_type = df.engine_type.str.strip().str.title()
# unifying missing values 
missing_values = ["Unknown", "Unk", "None", "Lr"]
df.engine_type = df.engine_type.replace(missing_values, pd.NA)

# fill the nulls
df.engine_type = df.engine_type.fillna("Unknown")
# df.engine_type.unique()

In [174]:
df.weather_condition.unique()
# unify the missiing values 
missing = ["Unk", "UNK"]
df.weather_condition = df.weather_condition.replace(missing, pd.NA)
df.weather_condition = df.weather_condition.fillna("Unknown")


In [175]:
df.number_of_engines.unique()
# airplane with zero engines don't exist lets remove them
df.number_of_engines = df.number_of_engines.replace(0, pd.NA)
df.number_of_engines = df.number_of_engines.fillna("Unknown")

In [176]:
df.purpose_of_flight.unique()
df.purpose_of_flight = df.purpose_of_flight.fillna("Unknown")
# df.purpose_of_flight.value_counts(dropna=False)

In [177]:
# filling nulls
df.broad_phase_of_flight = df.broad_phase_of_flight.fillna("Missing")
# df.broad_phase_of_flight.value_counts(dropna=False)

In [178]:
df.info()

<class 'pandas.DataFrame'>
Index: 17879 entries, 4150 to 88886
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype        
---  ------                  --------------  -----        
 0   event_id                17879 non-null  str          
 1   investigation_type      17879 non-null  str          
 2   accident_number         17879 non-null  str          
 3   event_date              17879 non-null  datetime64[s]
 4   location                17875 non-null  str          
 5   country                 17878 non-null  str          
 6   latitude                15981 non-null  object       
 7   longitude               15978 non-null  object       
 8   airport_code            11648 non-null  str          
 9   airport_name            11754 non-null  str          
 10  injury_severity         17162 non-null  str          
 11  aircraft_damage         17879 non-null  str          
 12  aircraft_category       17879 non-null  str          
 13  registration_n

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [179]:
# calculate % missing per column

missing_percent = df.isna().mean() * 100
missing_percent.sort_values(ascending=False).head()
# drop any column with more than 30% missing values 
df.info()

<class 'pandas.DataFrame'>
Index: 17879 entries, 4150 to 88886
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype        
---  ------                  --------------  -----        
 0   event_id                17879 non-null  str          
 1   investigation_type      17879 non-null  str          
 2   accident_number         17879 non-null  str          
 3   event_date              17879 non-null  datetime64[s]
 4   location                17875 non-null  str          
 5   country                 17878 non-null  str          
 6   latitude                15981 non-null  object       
 7   longitude               15978 non-null  object       
 8   airport_code            11648 non-null  str          
 9   airport_name            11754 non-null  str          
 10  injury_severity         17162 non-null  str          
 11  aircraft_damage         17879 non-null  str          
 12  aircraft_category       17879 non-null  str          
 13  registration_n

In [180]:
# filter based on a threshols of 14500 non-nulls
df = df.loc[:, df.count() > 14500]

In [181]:
df.isna().sum()

event_id                     0
investigation_type           0
accident_number              0
event_date                   0
location                     4
country                      1
latitude                  1898
longitude                 1901
injury_severity            717
aircraft_damage              0
aircraft_category            0
registration_number        164
make                         0
model                        0
amateur_built                0
number_of_engines            0
engine_type                  0
far_description            344
purpose_of_flight            0
total_fatal_injuries      2386
total_serious_injuries    2458
total_minor_injuries      2223
total_uninjured            587
weather_condition            0
broad_phase_of_flight        0
publication_date           788
total_passengers             0
total_serious_fatal       2582
injury_rate               3322
aircraft_destroyed           0
aircraft_type                0
dtype: int64

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [182]:
df.to_csv("data/cleaned_aviation_data.csv", index=False)